# Agronavis disease model v2 — trained on field photos

**Why this exists.** The current scanner (ResNet-18, 86 classes) was trained on PlantVillage-style photos: one leaf, plain background, studio light. A 2026 study tested the same architecture: **99.7% on lab photos, 27% on real field photos**. The model mostly learned the background. Rice, the crop that matters most in West Bengal, also has only a vague `diseased_rice` class.

**What this notebook does**
1. Gathers **field-photographed** datasets (rice, potato, tomato, maize…) into one label set built around Bengal's crops.
2. Fine-tunes a modern backbone (ConvNeXt-Nano, ImageNet-12k) with field-style augmentation and class balancing.
3. Tests it honestly, per source and *within the crop the farmer picked* (that is how the app now asks).
4. Calibrates the confidence (temperature scaling), so "70% sure" means about 70%.
5. Exports `disease_v2.onnx` + `labels.json` for `model/v2/`. The server picks them up automatically; no code change needed.

**Runtime:** Colab → *Runtime ▸ Change runtime type ▸ T4 GPU*. About 30–90 minutes, depending on how many datasets you add.

## 1. Setup

In [ ]:
!pip -q install timm onnx onnxruntime onnxscript scikit-learn matplotlib

import os, re, json, glob, random, zipfile, hashlib, shutil, datetime, collections
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from google.colab import drive
drive.mount('/content/drive')

# ── Settings ────────────────────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/agronavis/disease'   # everything lives here
RAW        = f'{DRIVE_ROOT}/raw'                          # one folder per dataset
OUT        = f'{DRIVE_ROOT}/out'                          # model + report land here
BACKBONE   = 'convnext_nano.in12k_ft_in1k'                # ~15.6M params, ~62 MB ONNX
IMG        = 224
EPOCHS     = 15
BATCH      = 48
LR         = 3e-4
SEED       = 7
# Crops the model learns. Anything else in the datasets is skipped.
INCLUDE_CROPS = ['rice', 'potato', 'tomato', 'corn', 'wheat', 'jute', 'mustard',
                 'bell_pepper', 'brinjal', 'chilli']

os.makedirs(RAW, exist_ok=True); os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)
assert device == 'cuda', 'Switch the runtime to a T4 GPU (Runtime ▸ Change runtime type).'

## 2. Get the datasets

Put each dataset in its own folder under `MyDrive/agronavis/disease/raw/`. A `.zip` dropped there is unzipped for you. Class names come from folder names, so any layout of *class folder → images* works.

| Folder name in `raw/` | Dataset | Crop | Why | License |
|---|---|---|---|---|
| `riceleafbd` | [RiceLeafBD](https://data.mendeley.com/datasets/kx9rx8p2mz) | rice | Field photos from phones: bacterial leaf blight, brown spot, tungro, healthy | CC BY 4.0 |
| `dhan_shomadhan` | [Dhan-Shomadhan](https://data.mendeley.com/datasets/znsxdctwtt) | rice | Bangladeshi local rice (same delta as Bengal): blast, brown spot, sheath blight, leaf scald, tungro | CC BY 4.0 |
| `potato_field` | [Potato Leaf Disease in Uncontrolled Environment](https://data.mendeley.com/datasets/ptz377bwb8) | potato | Field photos; *Phytophthora* = late blight | CC BY 4.0 |
| `plantdoc` | [PlantDoc](https://github.com/pratikkayal/PlantDoc-Dataset) (cloned below) | many | Real-world photos of tomato, potato, maize, pepper | CC BY 4.0 |
| `plantvillage` *(optional)* | PlantVillage (e.g. from Kaggle) | many | Lab photos — **capped** so they cannot dominate | check the copy you download |
| `own_photos` *(strongly recommended)* | Your own farmers' photos, `own_photos/<class folder>/…jpg` | any | **Test only, never trained on.** The only test that tells you how it does in Bengal | yours |

More rice classes later: add another folder (for example a hispa or false-smut set) and re-run. The label rules in step 3 decide what each folder means, and they print what they did.

In [ ]:
# PlantDoc is on GitHub, so it can be cloned directly.
import subprocess
if not os.path.exists(f'{RAW}/plantdoc'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1',
                    'https://github.com/pratikkayal/PlantDoc-Dataset', f'{RAW}/plantdoc'], check=True)

# Unzip anything uploaded as a zip.
for z in glob.glob(f'{RAW}/**/*.zip', recursive=True):
    target = z[:-4]
    if not os.path.exists(target):
        print('unzipping', os.path.relpath(z, RAW))
        with zipfile.ZipFile(z) as f:
            f.extractall(target)

# Training straight off Google Drive is slow (every image is a network read),
# so the datasets are copied to the Colab disk once.
SRC = '/content/raw'
if not os.path.exists(SRC):
    shutil.copytree(RAW, SRC, ignore=shutil.ignore_patterns('*.zip'))
print(sorted(os.listdir(SRC)))

## 3. One label set for all of them

Each dataset names its classes differently ("Steath Blight", "Tomato leaf late blight", "Corn_(maize)___Common_rust_"). These rules map a folder name, plus the dataset's crop where the folder doesn't say it, onto one key such as `rice_sheath_blight`. **Read the printed table**: every folder and what it became. Anything marked `SKIPPED` was not understood; add a rule if it should count.

In [ ]:
# Per-dataset settings. A dataset not listed here is read with no crop hint.
DATASETS = {
    'riceleafbd':     {'crop': 'rice'},
    'dhan_shomadhan': {'crop': 'rice'},
    'potato_field':   {'crop': 'potato'},
    'plantdoc':       {},
    'plantvillage':   {'cap_per_class': 250, 'lab': True},   # lab photos: capped
    'own_photos':     {'test_only': True},
}

CROP_WORDS = [
    ('bell_pepper', r'bell.?pepper|capsicum|pepper'),
    ('rice',        r'\brice\b|paddy|dhan'),
    ('potato',      r'potato'),
    ('tomato',      r'tomato'),
    ('corn',        r'corn|maize'),
    ('wheat',       r'wheat'),
    ('jute',        r'jute'),
    ('mustard',     r'mustard|rapeseed'),
    ('brinjal',     r'brinjal|eggplant|aubergine'),
    ('chilli',      r'chil+i|chilly'),
    ('apple', r'apple'), ('grape', r'grape'), ('cherry', r'cherry'), ('peach', r'peach'),
    ('strawberry', r'strawberr'), ('squash', r'squash'), ('soybean', r'soy'),
    ('blueberry', r'blueberr'), ('raspberry', r'raspberr'), ('orange', r'orange'),
]

# (condition key, pattern) per crop, first match wins. Order matters.
DISEASE_RULES = {
    'rice': [
        ('healthy', r'healthy|normal'),
        ('sheath_blight', r'sh?eath|steath'),
        ('bacterial_leaf_blight', r'bacteri|\bblb\b|leaf.?blight|^blight'),
        ('brown_spot', r'brown.?spot'),
        ('blast', r'blast'),
        ('tungro', r'tungro|turngo'),
        ('leaf_scald', r'scald|scaled'),
        ('hispa', r'hispa'),
        ('false_smut', r'false.?smut'),
        ('leaf_smut', r'leaf.?smut'),
    ],
    'potato': [
        ('healthy', r'healthy'),
        ('early_blight', r'early.?blight'),
        ('late_blight', r'late.?blight|phytophthora'),
        ('fungal_disease', r'fung'),
        ('bacterial_disease', r'bacteri'),
        ('virus', r'virus'),
        ('pest_damage', r'\bpests?\b|insect'),
        ('nematode', r'nematode'),
    ],
    'tomato': [
        ('healthy', r'healthy'),
        ('early_blight', r'early.?blight'),
        ('late_blight', r'late.?blight'),
        ('septoria_leaf_spot', r'septoria'),
        ('bacterial_spot', r'bacterial.?spot'),
        ('leaf_mold', r'mou?ld'),
        ('mosaic_virus', r'mosaic'),
        ('yellow_leaf_curl_virus', r'yellow|curl'),
        ('spider_mites', r'mite'),
        ('target_spot', r'target'),
    ],
    'corn': [
        ('healthy', r'healthy'),
        ('gray_leaf_spot', r'gr[ae]y|cercospora'),
        ('common_rust', r'rust'),
        ('northern_leaf_blight', r'blight'),
    ],
    'bell_pepper': [
        ('healthy', r'healthy'),
        ('bacterial_spot', r'spot'),
    ],
    'wheat': [
        ('healthy', r'healthy'),
        ('yellow_rust', r'yellow.?rust|stripe'),
        ('brown_rust', r'brown.?rust|leaf.?rust'),
        ('black_rust', r'black.?rust|stem.?rust'),
        ('blast', r'blast'),
        ('powdery_mildew', r'mildew'),
        ('septoria', r'septoria'),
        ('loose_smut', r'smut'),
    ],
}

CONDITION_NAMES = {
    'healthy': 'Healthy',
    'bacterial_leaf_blight': 'Bacterial Leaf Blight', 'brown_spot': 'Brown Spot', 'blast': 'Blast',
    'tungro': 'Tungro', 'sheath_blight': 'Sheath Blight', 'leaf_scald': 'Leaf Scald', 'hispa': 'Hispa',
    'false_smut': 'False Smut', 'leaf_smut': 'Leaf Smut',
    'early_blight': 'Early Blight', 'late_blight': 'Late Blight',
    'fungal_disease': 'Fungal disease (type not identified)',
    'bacterial_disease': 'Bacterial disease (type not identified)',
    'virus': 'Viral disease (type not identified)', 'pest_damage': 'Pest damage', 'nematode': 'Nematode damage',
    'septoria_leaf_spot': 'Septoria Leaf Spot', 'bacterial_spot': 'Bacterial Spot', 'leaf_mold': 'Leaf Mold',
    'mosaic_virus': 'Mosaic Virus', 'yellow_leaf_curl_virus': 'Yellow Leaf Curl Virus',
    'spider_mites': 'Spider Mites', 'target_spot': 'Target Spot',
    'gray_leaf_spot': 'Gray Leaf Spot', 'common_rust': 'Common Rust', 'northern_leaf_blight': 'Northern Leaf Blight',
    'yellow_rust': 'Yellow Rust', 'brown_rust': 'Brown Rust', 'black_rust': 'Black Rust',
    'powdery_mildew': 'Powdery Mildew', 'septoria': 'Septoria', 'loose_smut': 'Loose Smut',
}
CROP_NAMES = {'rice': 'Rice', 'potato': 'Potato', 'tomato': 'Tomato', 'corn': 'Corn', 'wheat': 'Wheat',
              'jute': 'Jute', 'mustard': 'Mustard', 'bell_pepper': 'Bell Pepper', 'brinjal': 'Brinjal',
              'chilli': 'Chilli'}

def norm(s):
    return re.sub(r'[_\-\(\)\.]+', ' ', s).lower().strip()

def crop_of(name, hint=None):
    n = norm(name)
    for crop, pat in CROP_WORDS:
        if re.search(pat, n):
            return crop
    return hint

def label_for(folder, hint=None):
    # (crop, condition) for a class folder name, or None if not understood.
    crop = crop_of(folder, hint)
    if crop is None or crop not in DISEASE_RULES:
        return None
    n = norm(folder)
    # Drop the crop word so "Potato leaf late blight" reads as "leaf late blight".
    for c, pat in CROP_WORDS:
        if c == crop:
            n = re.sub(pat, ' ', n).strip()
    for cond, pat in DISEASE_RULES[crop]:
        if re.search(pat, n):
            return crop, cond
    # PlantDoc names a healthy leaf just "<Crop> leaf".
    if re.fullmatch(r'(leaf|leaves)?', n.strip()):
        return crop, 'healthy'
    return None

IMAGE_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

In [ ]:
def csv_labels(root):
    # Some datasets ship flat image folders plus a CSV of labels (RiceLeafBD does).
    # Finds a column of image file names and a column of class names in any CSV.
    import pandas as pd
    found = {}
    for csv in glob.glob(f'{root}/**/*.csv', recursive=True):
        try:
            df = pd.read_csv(csv)
        except Exception:
            continue
        files = [c for c in df.columns if df[c].astype(str).str.lower().str.endswith(IMAGE_EXT).mean() > 0.8]
        texty = lambda c: pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object
        names = [c for c in df.columns if c not in files and texty(c) and 1 < df[c].nunique() <= 60]
        # Prefer a column that is called what it is.
        names.sort(key=lambda c: 0 if re.search(r'label|class|disease|category', str(c), re.I) else 1)
        if files and names:
            for f, lab in zip(df[files[0]].astype(str), df[names[0]].astype(str)):
                found[os.path.basename(f)] = lab
            print(f'   labels from {os.path.relpath(csv, SRC)}: {files[0]} → {names[0]}')
    return found

def label_from_path(path, root, hint, by_csv):
    # The class is the nearest folder (or CSV entry) that the rules understand,
    # so "Brown Spot/Field background/x.jpg" and "Field background/Brown Spot/x.jpg" both work.
    base = os.path.basename(path)
    if base in by_csv:
        lab = label_for(by_csv[base], hint)
        if lab: return lab, by_csv[base]
    d = os.path.dirname(path)
    for _ in range(3):
        if os.path.normpath(d) == os.path.normpath(root): break
        lab = label_for(os.path.basename(d), hint)
        if lab: return lab, os.path.basename(d)
        d = os.path.dirname(d)
    return None, os.path.basename(os.path.dirname(path))

def scan_dataset(name, cfg):
    root = f'{SRC}/{name}'
    rows, table = [], collections.Counter()
    by_csv = csv_labels(root)
    for path in glob.glob(f'{root}/**/*', recursive=True):
        if not path.lower().endswith(IMAGE_EXT):
            continue
        lab, folder = label_from_path(path, root, cfg.get('crop'), by_csv)
        key = f'{lab[0]}_{lab[1]}' if lab else None
        table[(folder, key if key and lab[0] in INCLUDE_CROPS else 'SKIPPED')] += 1
        if lab and lab[0] in INCLUDE_CROPS:
            rows.append({'path': path, 'key': key, 'crop': lab[0], 'cond': lab[1], 'source': name,
                         'lab': bool(cfg.get('lab')), 'test_only': bool(cfg.get('test_only'))})
    cap = cfg.get('cap_per_class')
    if cap:
        by = collections.defaultdict(list)
        for r in rows: by[r['key']].append(r)
        rows = [r for k in by for r in random.sample(by[k], min(cap, len(by[k])))]
    return rows, table

ALL = []
for name in sorted(os.listdir(SRC)):
    if not os.path.isdir(f'{SRC}/{name}'):
        continue
    rows, table = scan_dataset(name, DATASETS.get(name, {}))
    ALL += rows
    print(f'\n── {name}: {len(rows)} images used')
    for (folder, key), n in sorted(table.items()):
        print(f'   {folder[:45]:45s} → {key:35s} {n:6d}')

counts = collections.Counter(r['key'] for r in ALL if not r['test_only'])
print('\nTraining classes:', len(counts))
for k, n in sorted(counts.items()):
    print(f'  {k:40s} {n:6d}')

## 4. Split: train / validation / test

Split **inside each source** by class (70/15/15), so every source appears in the test set and we can report accuracy per source. Classes with fewer than 20 training images are dropped (too few to learn, too few to test). `own_photos` is test-only.

In [ ]:
MIN_IMAGES = 20
keep = {k for k, n in counts.items() if n >= MIN_IMAGES}
dropped = sorted(set(counts) - keep)
if dropped:
    print('Dropped (too few images):', dropped)

CLASSES = sorted(keep)
IDX = {k: i for i, k in enumerate(CLASSES)}
train, val, test = [], [], []
groups = collections.defaultdict(list)
for r in ALL:
    if r['key'] not in IDX:
        continue
    if r['test_only']:
        test.append(r); continue
    groups[(r['source'], r['key'])].append(r)
for g in groups.values():
    random.shuffle(g)
    n = len(g); a = int(n * 0.70); b = int(n * 0.85)
    train += g[:a]; val += g[a:b]; test += g[b:]
print(f'{len(CLASSES)} classes | train {len(train)} | val {len(val)} | test {len(test)}')

## 5. Data pipeline: augmentation that looks like a farmer's photo

Random crops and zoom, rotation, flips, light and colour shifts, blur and JPEG compression stand in for a hand-held phone in the field. Lab images get extra background and colour noise so the model cannot learn "plain background = some disease".

In [ ]:
import timm
from torchvision.transforms import v2 as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

cfg = timm.data.resolve_model_data_config(timm.create_model(BACKBONE, pretrained=False))
MEAN, STD = list(cfg['mean']), list(cfg['std'])

# Tensors from the start: JPEG re-compression needs uint8 tensors, and the rest work on them too.
train_tf = T.Compose([
    T.ToImage(),
    T.RandomResizedCrop(IMG, scale=(0.35, 1.0), ratio=(0.75, 1.33)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(p=0.2), T.RandomRotation(25),
    T.ColorJitter(0.35, 0.35, 0.3, 0.04), T.RandomGrayscale(0.03),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.25),
    T.RandomApply([T.JPEG((30, 90))], p=0.4),
    T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.ToImage(), T.Resize(IMG), T.CenterCrop(IMG),
                     T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])

class Leaves(Dataset):
    def __init__(self, rows, tf): self.rows, self.tf = rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        try:
            img = Image.open(r['path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG, IMG))
        return self.tf(img), IDX[r['key']]

# Balance: every class is seen about equally often, whatever its size.
freq = collections.Counter(r['key'] for r in train)
weights = [1.0 / freq[r['key']] for r in train]
sampler = WeightedRandomSampler(weights, num_samples=len(train), replacement=True)
dl_train = DataLoader(Leaves(train, train_tf), batch_size=BATCH, sampler=sampler, num_workers=2, drop_last=True)
dl_val   = DataLoader(Leaves(val, eval_tf), batch_size=BATCH, num_workers=2)
dl_test  = DataLoader(Leaves(test, eval_tf), batch_size=BATCH, num_workers=2)

## 6. Train

In [ ]:
from sklearn.metrics import f1_score

model = timm.create_model(BACKBONE, pretrained=True, num_classes=len(CLASSES), drop_path_rate=0.1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
steps = EPOCHS * len(dl_train)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler('cuda')

@torch.no_grad()
def predict(loader):
    model.eval(); logits, ys = [], []
    for x, y in loader:
        with torch.autocast('cuda', dtype=torch.float16):
            logits.append(model(x.to(device)).float().cpu())
        ys.append(y)
    return torch.cat(logits), torch.cat(ys)

best, best_state = -1, None
for epoch in range(EPOCHS):
    model.train(); total = 0
    for x, y in dl_train:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    lv, yv = predict(dl_val)
    f1 = f1_score(yv, lv.argmax(1), average='macro')
    acc = (lv.argmax(1) == yv).float().mean().item()
    print(f'epoch {epoch+1:2d} | loss {total/len(dl_train):.3f} | val acc {acc:.3f} | val macro-F1 {f1:.3f}')
    if f1 > best:
        best, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
print('best val macro-F1', round(best, 3))

## 7. Calibrate the confidence

A network trained like this is usually over-confident. One number, the temperature *T*, fixes most of it: the logits are divided by *T* before softmax. *T* is fitted on the validation set and saved in `labels.json`; the server applies it.

In [ ]:
lv, yv = predict(dl_val)
logT = torch.zeros(1, requires_grad=True)
lbfgs = torch.optim.LBFGS([logT], lr=0.05, max_iter=200)
def closure():
    lbfgs.zero_grad(); loss = F.cross_entropy(lv / logT.exp(), yv); loss.backward(); return loss
lbfgs.step(closure)
TEMPERATURE = float(logT.exp())

def ece(probs, y, bins=15):
    conf, pred = probs.max(1); acc = (pred == y).float(); e = torch.zeros(1)
    for lo in torch.linspace(0, 1, bins + 1)[:-1]:
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any(): e += m.float().mean() * (conf[m].mean() - acc[m].mean()).abs()
    return float(e)
print(f'temperature {TEMPERATURE:.2f} | ECE before {ece(lv.softmax(1), yv):.3f} → after {ece((lv / TEMPERATURE).softmax(1), yv):.3f}')

## 8. Test honestly

Three numbers matter more than overall accuracy:
- **Per source.** Field sets versus lab sets. A big gap means it still leans on backgrounds.
- **Within the crop.** The app now tells the model the crop, so this is the accuracy a farmer actually gets.
- **`own_photos`**, if present. Photos from Bengal fields that it never trained on. Treat this as the real score.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

lt, yt = predict(dl_test)
pt = (lt / TEMPERATURE).softmax(1)
pred = pt.argmax(1)
crop_of_class = [k.split('_')[0] if not k.startswith('bell_pepper') else 'bell_pepper' for k in CLASSES]

# Within-crop: restrict each prediction to the true crop's classes, as the app does.
within = []
for i in range(len(yt)):
    c = crop_of_class[yt[i]]
    idx = [j for j, cc in enumerate(crop_of_class) if cc == c]
    within.append(idx[int(pt[i, idx].argmax())] == int(yt[i]))
within = np.array(within)

lines = [f'# Disease model v2 — test report ({datetime.date.today()})', '',
         f'Backbone `{BACKBONE}`, {len(CLASSES)} classes, temperature {TEMPERATURE:.2f}.', '',
         f'- Overall accuracy: **{(pred == yt).float().mean():.3f}**',
         f'- Within the farmer\'s crop: **{within.mean():.3f}**',
         f'- Macro-F1: **{f1_score(yt, pred, average="macro"):.3f}**', '', '## Per source', '',
         '| Source | Images | Accuracy | Within crop |', '|---|---|---|---|']
sources = np.array([r['source'] for r in test])
for s in sorted(set(sources)):
    m = sources == s
    lines.append(f'| {s} | {m.sum()} | {(pred.numpy()[m] == yt.numpy()[m]).mean():.3f} | {within[m].mean():.3f} |')
lines += ['', '## Per class', '', '```',
          classification_report(yt, pred, labels=list(range(len(CLASSES))), target_names=CLASSES, zero_division=0),
          '```']
REPORT = '\n'.join(lines)
print(REPORT)

cm = confusion_matrix(yt, pred, labels=list(range(len(CLASSES))))
plt.figure(figsize=(12, 10)); plt.imshow(cm, cmap='Greens')
plt.xticks(range(len(CLASSES)), CLASSES, rotation=90, fontsize=7); plt.yticks(range(len(CLASSES)), CLASSES, fontsize=7)
plt.xlabel('predicted'); plt.ylabel('true'); plt.tight_layout(); plt.savefig(f'{OUT}/confusion.png', dpi=150)

## 9. Export for the server

ONNX with input `image` `[N, 3, 224, 224]` and output `logits`, the names `src/modules/crops/diagnose.service.ts` reads. The export is checked against PyTorch before it is saved.

In [ ]:
import onnxruntime as ort
model.eval().cpu()
dummy = torch.randn(1, 3, IMG, IMG)
onnx_path = f'{OUT}/disease_v2.onnx'
torch.onnx.export(model, dummy, onnx_path, input_names=['image'], output_names=['logits'],
                  dynamic_axes={'image': {0: 'n'}, 'logits': {0: 'n'}}, opset_version=17, dynamo=False)

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
x = torch.randn(2, 3, IMG, IMG)
diff = np.abs(sess.run(['logits'], {'image': x.numpy()})[0] - model(x).detach().numpy()).max()
print('max |onnx - torch| =', float(diff)); assert diff < 1e-3

version = f'v2-{datetime.date.today():%Y.%m.%d}'
labels = {
    'version': version, 'backbone': BACKBONE, 'temperature': round(TEMPERATURE, 4), 'size': IMG,
    'mean': MEAN, 'std': STD,
    'classes': [{'key': k, 'crop': CROP_NAMES.get(k.split('_')[0] if not k.startswith('bell_pepper') else 'bell_pepper', k),
                 'condition': CONDITION_NAMES.get(k.split('_', 1)[1] if not k.startswith('bell_pepper') else k[len('bell_pepper_'):], k),
                 'healthy': k.endswith('_healthy')} for k in CLASSES],
    'metrics': {'test_accuracy': round(float((pred == yt).float().mean()), 4),
                'within_crop_accuracy': round(float(within.mean()), 4),
                'macro_f1': round(float(f1_score(yt, pred, average='macro')), 4)},
    'sources': sorted({r['source'] for r in ALL}),
}
json.dump(labels, open(f'{OUT}/labels.json', 'w'), indent=2)
open(f'{OUT}/report.md', 'w').write(REPORT)
print(f'Saved to {OUT}:', os.listdir(OUT), '| ONNX MB:', round(os.path.getsize(onnx_path) / 1e6, 1))

## 10. Ship it

1. Download `disease_v2.onnx` and `labels.json` from `MyDrive/agronavis/disease/out/`.
2. Put them in this repo (backend-pro) at `model/v2/` (create the folder).
3. Commit and push. Render redeploys, and the server logs `Disease model loaded { version: 'v2-…' }`. If either file is missing or they disagree, it falls back to v1 by itself.
4. Run `npm run seed:diseases` once, so the reference library has the new classes.
5. Keep `report.md` with the release. If *within crop* on `own_photos` is under ~0.8, collect more of your own labelled photos and train again. The scans farmers file under a disease in the app (`crop_scans`) are the cheapest source, once an agronomist has checked them.

**Licences.** RiceLeafBD, Dhan-Shomadhan, the potato set and PlantDoc are CC BY 4.0: credit them in the app's About screen or on the website. Check the licence of any PlantVillage copy you add.